This notebook is for combining tables and creating features

In [4]:
import pandas as pd

In [5]:
# Load tables here
team_stats15 = pd.read_csv('../data/raw/stats_team_week_2015.csv')
team_stats16 = pd.read_csv('../data/raw/stats_team_week_2016.csv')
team_stats17 = pd.read_csv('../data/raw/stats_team_week_2017.csv')
team_stats18 = pd.read_csv('../data/raw/stats_team_week_2018.csv')
team_stats19 = pd.read_csv('../data/raw/stats_team_week_2019.csv')
team_stats20 = pd.read_csv('../data/raw/stats_team_week_2020.csv')
team_stats21 = pd.read_csv('../data/raw/stats_team_week_2021.csv')
team_stats22 = pd.read_csv('../data/raw/stats_team_week_2022.csv')
team_stats23 = pd.read_csv('../data/raw/stats_team_week_2023.csv')
team_stats24 = pd.read_csv('../data/raw/stats_team_week_2024.csv')
games_table = pd.read_csv('../data/raw/games.csv')

In [6]:
# Combine the tables
team_stats_all = pd.concat([team_stats15, team_stats16, team_stats17, team_stats18, team_stats19, team_stats20, team_stats21,
                           team_stats22, team_stats23, team_stats24])
team_stats_all

,season,week,team,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,...,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance
0,2015,1,ARI,REG,NO,19,32,307,3,0,...,4,4,0,0,1.0,0,0,0,0,0
1,2015,1,ATL,REG,PHI,23,34,298,2,2,...,2,2,0,0,1.0,0,0,0,0,0
2,2015,1,BAL,REG,DEN,18,32,117,0,2,...,1,1,0,0,1.0,0,0,0,0,0
3,2015,1,BUF,REG,IND,14,19,195,1,0,...,3,3,0,0,1.0,0,0,0,0,0
4,2015,1,CAR,REG,JAX,18,31,175,1,1,...,2,2,0,0,1.0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565,2024,21,KC,POST,BUF,18,26,245,1,0,...,3,3,0,0,1.0,0,0,0,0,0
566,2024,21,PHI,POST,WAS,20,28,246,1,0,...,7,7,0,0,1.0,0,0,0,0,0
567,2024,21,WAS,POST,PHI,30,49,278,1,1,...,0,0,0,0,NaN,0,0,0,0,0
568,2024,22,KC,POST,PHI,21,32,257,3,2,...,0,0,0,0,NaN,0,0,0,0,0


In [7]:
unique_seasons = team_stats_all['season'].value_counts()
unique_seasons = unique_seasons.sort_values(ascending=True)
unique_seasons

season
2015    534
2016    534
2017    534
2018    534
2019    534
2020    538
2022    568
2021    570
2023    570
2024    570
Name: count, dtype: int64

In [8]:
def retrieve_stats(season, week, team):
    # retrieves all stats for a team for each game they have played up until the week that is inputed
    games = team_stats_all.loc[team_stats_all['season'] == season]
    games = games.loc[games['week'] < week]
    games = games.loc[games['team'] == team]
    return games

retrieve_stats(2022, 3, 'BUF')

,season,week,team,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,...,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance
3,2022,1,BUF,REG,LA,26,31,297,3,2,...,4,4,0,0,1.0,0,0,0,0,0
35,2022,2,BUF,REG,TEN,26,40,317,4,0,...,5,5,0,0,1.0,0,0,0,0,0


In [9]:
def retrieve_opponent_stats(season, week, team):
    # returns the opponent stats from the game in which the play the team that is an input
    game = team_stats_all.loc[team_stats_all['season'] == season]
    game = game.loc[game['week'] == week]
    game_table = game.loc[game['team'] == team]
    opponent = game_table['opponent_team'].iloc[0]
    opponent_stats_game = game.loc[game['team'] == opponent]
    opponent_stats = {'passing_yards': opponent_stats_game['passing_yards'].iloc[0], 'rushing_yards': opponent_stats_game['rushing_yards'].iloc[0]}
    return opponent_stats

retrieve_opponent_stats(2015, 8, 'PIT')

{'passing_yards': np.int64(231), 'rushing_yards': np.int64(78)}

In [10]:
def retrieve_game_stats(season, week, team):
    # retrieves game stats from a certain game (points scored for both teams)
    game = games_table.loc[(games_table['season'] == season) & (games_table['week'] == week) 
                           & ((games_table['home_team'] == team) | (games_table['away_team'] == team))]
    if game['home_team'].iloc[0] == team:
        points_scored = game['home_score'].iloc[0]
        points_allowed = game['away_score'].iloc[0]
    else:
        points_scored = game['away_score'].iloc[0]
        points_allowed = game['home_score'].iloc[0]
    return {
        'points_scored': points_scored,
        'points_allowed': points_allowed
    }

retrieve_game_stats(1999, 2, 'ATL')

{'points_scored': np.float64(7.0), 'points_allowed': np.float64(24.0)}

In [11]:
def calculate_team_statistics(previous_games):
    # calculates team statistics needed for the ML model
    points_scored = 0
    points_allowed = 0
    passing_yards_allowed = 0
    rushing_yards_allowed = 0
    if len(previous_games) == 0:
        avg_points = 0
        passing_avg = 0
        rushing_avg = 0
        avg_turnovers = 0
        avg_points_allowed = 0
        penalties_avg = 0
        avg_passing_yards_allowed = 0
        avg_rushing_yards_allowed = 0
        def_sacks_per_game = 0
        def_int_per_game = 0
        fg_pct = 0
    else:
        for i in range(len(previous_games)):
            season = previous_games['season'].iloc[i]
            week = previous_games['week'].iloc[i]
            team = previous_games['team'].iloc[i]
            opponent_stats = retrieve_opponent_stats(season, week, team)
            game_stats = retrieve_game_stats(season, week, team)
            points_scored += game_stats['points_scored']
            points_allowed += game_stats['points_allowed']
            passing_yards_allowed += opponent_stats['passing_yards']
            rushing_yards_allowed += opponent_stats['rushing_yards']
        
        avg_points = points_scored / len(previous_games)
        avg_points_allowed = points_allowed / len(previous_games)
        avg_passing_yards_allowed = passing_yards_allowed / len(previous_games)
        avg_rushing_yards_allowed = rushing_yards_allowed / len(previous_games)
        passing_avg = previous_games['passing_yards'].mean()
        rushing_avg = previous_games['rushing_yards'].mean()
        avg_turnovers = previous_games['passing_interceptions'].mean() + previous_games['sack_fumbles_lost'].mean() + previous_games['rushing_fumbles_lost'].mean() + previous_games['receiving_fumbles_lost'].mean()
        penalties_avg = previous_games['penalties'].mean()
        def_sacks_per_game = previous_games['def_sacks'].mean()
        def_int_per_game = previous_games['def_interceptions'].mean()
        fg_made = previous_games['fg_made'].sum()
        fg_att = previous_games['fg_att'].sum()
        if fg_att == 0:
            fg_pct = 0
        else:
            fg_pct = fg_made / fg_att
    
    return {
        'avg_points': round(avg_points, 2),
        'passing_avg': round(passing_avg, 2),
        'rushing_avg': round(rushing_avg, 2),
        'avg_turnovers': round(avg_turnovers, 2),
        'avg_points_allowed': round(avg_points_allowed, 2),
        'penalties_avg': round(penalties_avg, 2),
        'avg_passing_yards_allowed': round(avg_passing_yards_allowed, 2),
        'avg_rushing_yards_allowed': round(avg_rushing_yards_allowed, 2),
        'def_sacks_per_game': round(def_sacks_per_game, 2),
        'def_int_per_game': round(def_int_per_game, 2),
        'fg_pct': round(fg_pct, 2)
    }

calculate_team_statistics(retrieve_stats(2015, 8, 'CIN'))

{'avg_points': np.float64(30.33),
 'passing_avg': np.float64(293.5),
 'rushing_avg': np.float64(122.17),
 'avg_turnovers': np.float64(1.0),
 'avg_points_allowed': np.float64(20.33),
 'penalties_avg': np.float64(7.5),
 'avg_passing_yards_allowed': np.float64(278.0),
 'avg_rushing_yards_allowed': np.float64(109.17),
 'def_sacks_per_game': np.float64(2.83),
 'def_int_per_game': np.float64(0.83),
 'fg_pct': np.float64(0.78)}

In [12]:
def find_game(season, week, home_team, away_team):
    game = games_table.loc[(games_table['season'] == season) & (games_table['week'] == week) & (games_table['home_team'] == home_team) & (games_table['away_team'] == away_team)].iloc[0]
    return game
def build_ml_row(season, week, home_team, away_team):
    home_prev_games = retrieve_stats(season, week, home_team)
    away_prev_games = retrieve_stats(season, week, away_team)
    home_stats = calculate_team_statistics(home_prev_games)
    away_stats = calculate_team_statistics(away_prev_games)
    game = find_game(season, week, home_team, away_team)
    home_rest = game['home_rest']
    away_rest = game['away_rest']
    home_score = game['home_score']
    away_score = game['away_score']
    if home_score > away_score:
        home_win = 1
    elif away_score > home_score:
        home_win = 0
    else:
        return None
    return {
        'season': season,
        'week': week,
        'home_team': home_team,
        'away_team': away_team,
        'home_rest': home_rest,
        'away_rest': away_rest,
        'home_avg_points': home_stats['avg_points'],
        'home_passing_avg': home_stats['passing_avg'],
        'home_rushing_avg': home_stats['rushing_avg'],
        'home_avg_turnovers': home_stats['avg_turnovers'],
        'home_avg_points_allowed': home_stats['avg_points_allowed'],
        'home_penalties_avg': home_stats['penalties_avg'],
        'home_avg_passing_yards_allowed': home_stats['avg_passing_yards_allowed'],
        'home_avg_rushing_yards_allowed': home_stats['avg_rushing_yards_allowed'],
        'home_def_sacks_per_game': home_stats['def_sacks_per_game'],
        'home_def_int_per_game': home_stats['def_int_per_game'],
        'home_fg_pct': home_stats['fg_pct'],
        'away_avg_points': away_stats['avg_points'],
        'away_passing_avg': away_stats['passing_avg'],
        'away_rushing_avg': away_stats['rushing_avg'],
        'away_avg_turnovers': away_stats['avg_turnovers'],
        'away_avg_points_allowed': away_stats['avg_points_allowed'],
        'away_penalties_avg': away_stats['penalties_avg'],
        'away_avg_passing_yards_allowed': away_stats['avg_passing_yards_allowed'],
        'away_avg_rushing_yards_allowed': away_stats['avg_rushing_yards_allowed'],
        'away_def_sacks_per_game': away_stats['def_sacks_per_game'],
        'away_def_int_per_game': away_stats['def_int_per_game'],
        'away_fg_pct': away_stats['fg_pct'],
        'home_win': home_win
    }
build_ml_row(2022, 11, 'PIT', 'CIN')


{'season': 2022,
 'week': 11,
 'home_team': 'PIT',
 'away_team': 'CIN',
 'home_rest': np.int64(7),
 'away_rest': np.int64(14),
 'home_avg_points': np.float64(15.56),
 'home_passing_avg': np.float64(218.11),
 'home_rushing_avg': np.float64(108.44),
 'home_avg_turnovers': np.float64(1.22),
 'home_avg_points_allowed': np.float64(23.0),
 'home_penalties_avg': np.float64(6.0),
 'home_avg_passing_yards_allowed': np.float64(275.78),
 'home_avg_rushing_yards_allowed': np.float64(108.0),
 'home_def_sacks_per_game': np.float64(1.89),
 'home_def_int_per_game': np.float64(1.11),
 'home_fg_pct': np.float64(0.73),
 'away_avg_points': np.float64(25.33),
 'away_passing_avg': np.float64(286.67),
 'away_rushing_avg': np.float64(98.78),
 'away_avg_turnovers': np.float64(1.0),
 'away_avg_points_allowed': np.float64(20.56),
 'away_penalties_avg': np.float64(4.56),
 'away_avg_passing_yards_allowed': np.float64(215.0),
 'away_avg_rushing_yards_allowed': np.float64(118.78),
 'away_def_sacks_per_game': np.floa

In [21]:
#Looping through eligible games
eligible_games = games_table.loc[(games_table['game_type'] == 'REG')]
eligible_games = eligible_games.loc[(eligible_games['season'] <= 2024) & (eligible_games['season'] >= 2015)]
eligible_games = eligible_games.loc[(eligible_games['week'] > 1)]
ml_rows = []
for index, row in eligible_games.iterrows():
    ml_row = build_ml_row(row['season'], row['week'], row['home_team'], row['away_team'])
    if ml_row is None:
        continue
    else:
        ml_rows.append(ml_row)
ml_table = pd.DataFrame(data=ml_rows)

In [26]:
print(ml_table.shape)
print(ml_table.columns)

(2458, 29)
Index(['season', 'week', 'home_team', 'away_team', 'home_rest', 'away_rest',
       'home_avg_points', 'home_passing_avg', 'home_rushing_avg',
       'home_avg_turnovers', 'home_avg_points_allowed', 'home_penalties_avg',
       'home_avg_passing_yards_allowed', 'home_avg_rushing_yards_allowed',
       'home_def_sacks_per_game', 'home_def_int_per_game', 'home_fg_pct',
       'away_avg_points', 'away_passing_avg', 'away_rushing_avg',
       'away_avg_turnovers', 'away_avg_points_allowed', 'away_penalties_avg',
       'away_avg_passing_yards_allowed', 'away_avg_rushing_yards_allowed',
       'away_def_sacks_per_game', 'away_def_int_per_game', 'away_fg_pct',
       'home_win'],
      dtype='str')


In [25]:
ml_table.head()

,season,week,home_team,away_team,home_rest,away_rest,home_avg_points,home_passing_avg,home_rushing_avg,home_avg_turnovers,...,away_rushing_avg,away_avg_turnovers,away_avg_points_allowed,away_penalties_avg,away_avg_passing_yards_allowed,away_avg_rushing_yards_allowed,away_def_sacks_per_game,away_def_int_per_game,away_fg_pct,home_win
0,2015,2,KC,DEN,4,4,27.0,243.0,97.0,0.0,...,69.0,1.0,13.0,8.0,117.0,73.0,2.0,2.0,1.0,0
1,2015,2,BUF,NE,7,10,27.0,195.0,147.0,0.0,...,80.0,0.0,21.0,7.0,351.0,134.0,3.0,1.0,0.0,0
2,2015,2,CAR,HOU,7,7,20.0,175.0,105.0,1.0,...,98.0,2.0,27.0,6.0,243.0,97.0,2.0,0.0,1.0,1
3,2015,2,CHI,ARI,7,7,23.0,225.0,189.0,1.0,...,120.0,1.0,19.0,5.0,355.0,54.0,2.0,1.0,1.0,0
4,2015,2,CIN,SD,7,7,33.0,269.0,127.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [ ]:
ml_table.duplicated().sum()

np.int64(0)

In [28]:
ml_table.isnull().sum()

season                            0
week                              0
home_team                         0
away_team                         0
home_rest                         0
away_rest                         0
home_avg_points                   0
home_passing_avg                  0
home_rushing_avg                  0
home_avg_turnovers                0
home_avg_points_allowed           0
home_penalties_avg                0
home_avg_passing_yards_allowed    0
home_avg_rushing_yards_allowed    0
home_def_sacks_per_game           0
home_def_int_per_game             0
home_fg_pct                       0
away_avg_points                   0
away_passing_avg                  0
away_rushing_avg                  0
away_avg_turnovers                0
away_avg_points_allowed           0
away_penalties_avg                0
away_avg_passing_yards_allowed    0
away_avg_rushing_yards_allowed    0
away_def_sacks_per_game           0
away_def_int_per_game             0
away_fg_pct                 

In [29]:
ml_table['home_win'].value_counts()

home_win
1    1350
0    1108
Name: count, dtype: int64

In [32]:
print(ml_table['week'].min())
print(ml_table['season'].min())
print(ml_table['season'].max())

2
2015
2024


In [33]:
#Saving table to the processed data folder
ml_table.to_csv('../data/processed/ml_training_data.csv')